
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lecture - AI Search on Databricks
## Overview
This lecture covers the theory behind AI Search before you build one in the next demo. We'll explore how embeddings and similarity search power retrieval, how Knowledge Assistants use AI Search under the hood, and when to choose managed RAG vs. building your own. We'll also cover how Unity Catalog governs the entire pipeline.

## Learning Objectives
By the end of this lecture, you will be able to:
1. Explain how embeddings transform text into searchable vectors
2. Describe how cosine similarity drives retrieval ranking
3. Compare managed RAG (Knowledge Assistants) and custom RAG approaches
4. Explain how Unity Catalog governs AI Search indexes

## A. Embeddings and Similarity

### A1. What Are Embeddings?

An **embedding** is a list of numbers (a vector) that captures the meaning of a piece of text. Texts with similar meanings produce vectors that are close together in mathematical space.

For example, "cozy apartment near the park" and "comfortable flat close to green space" would have similar embeddings, even though they use completely different words.

For example, when creating a AI Search index with `embedding_model_endpoint_name="databricks-gte-large-en"`, Databricks calls this model on every chunk in your table to produce these vectors. Now let's understand how those vectors are used to find relevant results.

**Key considerations when working with embeddings:**
- **Use the same model** for indexing documents and processing queries, as this ensures they exist in the same vector space
- **Respect context windows** - embedding models have token limits (e.g., 512 tokens). Text beyond the limit is silently truncated
- **Dimension trade-offs** - higher dimensions capture more nuance but cost more to store and search

### A2. How Similarity Search Works

When you search a vector index, the system:
1. Converts your query text into an embedding vector
2. Finds the closest vectors in the index (using cosine similarity or other distance metrics)
3. Returns the corresponding text chunks, ranked by relevance

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 680 200" role="img" style="font-family: sans-serif;">
  <title>Similarity Search Flow</title>
  <defs>
    <marker id="ss-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>

  <!-- Query -->
  <rect x="10" y="60" width="130" height="60" rx="8" fill="#ffffff" stroke="#1B3139" stroke-width="1.5"/>
  <text x="75" y="84" text-anchor="middle" font-size="17" font-weight="600" fill="#0b2026">Query Text</text>
  <text x="75" y="102" text-anchor="middle" font-size="15" fill="#618794">"pet-friendly?"</text>

  <path d="M140 90 L180 90" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#ss-arrow)"/>

  <!-- Embed -->
  <rect x="185" y="60" width="130" height="60" rx="8" fill="#2272B4" fill-opacity="0.12" stroke="#2272B4" stroke-width="1.5"/>
  <text x="250" y="84" text-anchor="middle" font-size="17" font-weight="600" fill="#0b2026">Embed</text>
  <text x="250" y="102" text-anchor="middle" font-size="15" fill="#618794">[0.23, -0.41, ...]</text>

  <path d="M315 90 L355 90" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#ss-arrow)"/>

  <!-- Search -->
  <rect x="360" y="60" width="130" height="60" rx="8" fill="#00A972" fill-opacity="0.12" stroke="#00A972" stroke-width="1.5"/>
  <text x="425" y="84" text-anchor="middle" font-size="17" font-weight="600" fill="#0b2026">Find Nearest</text>
  <text x="425" y="102" text-anchor="middle" font-size="15" fill="#618794">Cosine similarity</text>

  <path d="M490 90 L530 90" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#ss-arrow)"/>

  <!-- Results -->
  <rect x="535" y="40" width="130" height="100" rx="8" fill="#F9F7F4" stroke="#1B3139" stroke-width="1.5"/>
  <text x="600" y="64" text-anchor="middle" font-size="17" font-weight="600" fill="#0b2026">Top Results</text>
  <rect x="547" y="76" width="106" height="20" rx="4" fill="#00A972" fill-opacity="0.2"/>
  <text x="600" y="90" text-anchor="middle" font-size="14" fill="#0b2026">Chunk 1 (0.94)</text>
  <rect x="547" y="100" width="106" height="20" rx="4" fill="#00A972" fill-opacity="0.12"/>
  <text x="600" y="114" text-anchor="middle" font-size="14" fill="#0b2026">Chunk 2 (0.87)</text>
  <rect x="547" y="124" width="106" height="10" rx="4" fill="#00A972" fill-opacity="0.06"/>
</svg>
</div>

Databricks uses **Approximate Nearest Neighbors (ANN)** algorithms for fast retrieval at scale, trading a small amount of precision for dramatic speed gains.

#### How Cosine Similarity Works

Cosine similarity measures the angle between two vectors. When vectors point in similar directions, the angle is small and the similarity score is high (close to 1.0). When they point in different directions, the score drops toward 0.

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 680 260" role="img" style="font-family: sans-serif;">
  <title>Cosine Similarity - vectors in 2D space</title>
  <desc>Two diagrams showing high similarity (small angle between vectors) and low similarity (large angle between vectors).</desc>
  <defs>
    <marker id="v-arrow-blue" viewBox="0 0 10 10" refX="0" refY="5" markerWidth="14" markerHeight="14" markerUnits="userSpaceOnUse" orient="auto">
      <path d="M0 1L10 5L0 9Z" fill="#2272B4"/>
    </marker>
    <marker id="v-arrow-green" viewBox="0 0 10 10" refX="0" refY="5" markerWidth="14" markerHeight="14" markerUnits="userSpaceOnUse" orient="auto">
      <path d="M0 1L10 5L0 9Z" fill="#00A972"/>
    </marker>
    <marker id="v-arrow-red" viewBox="0 0 10 10" refX="0" refY="5" markerWidth="14" markerHeight="14" markerUnits="userSpaceOnUse" orient="auto">
      <path d="M0 1L10 5L0 9Z" fill="#98102A"/>
    </marker>
  </defs>

  <!-- LEFT: High similarity -->
  <rect x="15" y="10" width="310" height="240" rx="12" fill="#F9F7F4" stroke="#1B3139" stroke-width="1" stroke-opacity="0.3"/>
  <text x="170" y="36" text-anchor="middle" font-size="17" font-weight="700" fill="#00A972">High Similarity (0.95)</text>

  <!-- Axes -->
  <line x1="60" y1="210" x2="280" y2="210" stroke="#618794" stroke-width="1"/>
  <line x1="60" y1="210" x2="60" y2="50" stroke="#618794" stroke-width="1"/>

  <!-- Vector A (query) -->
  <line x1="60" y1="210" x2="260" y2="80" stroke="#2272B4" stroke-width="2.5" marker-end="url(#v-arrow-blue)"/>
  <text x="275" y="72" font-size="16" font-weight="600" fill="#2272B4">Query</text>

  <!-- Vector B (document) -->
  <line x1="60" y1="210" x2="270" y2="100" stroke="#00A972" stroke-width="2.5" marker-end="url(#v-arrow-green)"/>
  <text x="230" y="135" font-size="16" font-weight="600" fill="#00A972">Doc chunk</text>

  <!-- Angle arc: from point on Query vector to point on Doc vector, radius 80 -->
  <path d="M127 166 A 80 80 0 0 1 131 173" fill="none" stroke="#FF5F46" stroke-width="2"/>
  <text x="140" y="185" font-size="15" fill="#FF5F46">small angle</text>

  <!-- RIGHT: Low similarity -->
  <rect x="355" y="10" width="310" height="240" rx="12" fill="#F9F7F4" stroke="#1B3139" stroke-width="1" stroke-opacity="0.3"/>
  <text x="510" y="36" text-anchor="middle" font-size="17" font-weight="700" fill="#98102A">Low Similarity (0.25)</text>

  <!-- Axes -->
  <line x1="400" y1="210" x2="620" y2="210" stroke="#618794" stroke-width="1"/>
  <line x1="400" y1="210" x2="400" y2="50" stroke="#618794" stroke-width="1"/>

  <!-- Vector A (query) -->
  <line x1="400" y1="210" x2="600" y2="80" stroke="#2272B4" stroke-width="2.5" marker-end="url(#v-arrow-blue)"/>
  <text x="615" y="72" font-size="16" font-weight="600" fill="#2272B4">Query</text>

  <!-- Vector B (document) - pointing mostly horizontal -->
  <line x1="400" y1="210" x2="610" y2="190" stroke="#98102A" stroke-width="2.5" marker-end="url(#v-arrow-red)"/>
  <text x="565" y="178" font-size="16" font-weight="600" fill="#98102A">Doc chunk</text>

  <!-- Angle arc: from point on Query vector to point on Doc vector, radius 70 -->
  <path d="M459 172 A 70 70 0 0 1 470 203" fill="none" stroke="#FF5F46" stroke-width="2"/>
  <text x="480" y="192" font-size="15" fill="#FF5F46">large angle</text>
</svg>
</div>

This is exactly what happens when you run `similarity_search` on a AI Search index. Your query text is embedded, compared against all vectors in the index, and the top matches are returned with their similarity scores. You'll see this in action in the next demo.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Learn More</strong>
<p style="margin: 8px 0 0 0; color: #333;">
See the full AI Search documentation (<a href="https://docs.databricks.com/aws/en/ai-search/ai-search" style="color: #1976d2;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/ai-search/ai-search" style="color: #1976d2;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/ai-search/ai-search" style="color: #1976d2;">GCP</a>) for details on endpoints, indexes, and APIs.
</p>
</div>

## B. Managed vs. Custom RAG

### B1. Two Paths to RAG

Now that you've built the full custom pipeline (parse, chunk, index, search), let's zoom out. Databricks provides two complementary approaches to building retrieval systems:

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 780 320" role="img" style="font-family: sans-serif;">
  <title>Managed RAG vs Custom RAG: two paths from documents to answers</title>
  <defs>
    <marker id="rag-arrow-green" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#00A972" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
    <marker id="rag-arrow-blue" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#2272B4" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>

  <!-- Documents (shared start) -->
  <rect x="10" y="110" width="130" height="90" rx="8" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/>
  <text x="75" y="140" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">Documents</text>
  <text x="75" y="160" text-anchor="middle" font-size="13" fill="#618794">PDFs, tables,</text>
  <text x="75" y="176" text-anchor="middle" font-size="13" fill="#618794">UC Volumes</text>

  <!-- Fork arrows -->
  <path d="M140 135 L210 65" fill="none" stroke="#00A972" stroke-width="1.8" marker-end="url(#rag-arrow-green)"/>
  <path d="M140 175 L210 235" fill="none" stroke="#2272B4" stroke-width="1.8" marker-end="url(#rag-arrow-blue)"/>

  <!-- MANAGED PATH (top) -->
  <rect x="215" y="15" width="310" height="100" rx="10" fill="#00A972" fill-opacity="0.08" stroke="#00A972" stroke-width="1.5"/>
  <text x="370" y="42" text-anchor="middle" font-size="17" font-weight="700" fill="#00A972">Managed RAG</text>
  <text x="370" y="62" text-anchor="middle" font-size="14" fill="#0b2026">Knowledge Assistant</text>
  <text x="370" y="82" text-anchor="middle" font-size="13" fill="#618794">Parsing, chunking, embedding,</text>
  <text x="370" y="98" text-anchor="middle" font-size="13" fill="#618794">indexing, serving are automatically implemented</text>

  <!-- CUSTOM PATH (bottom) -->
  <rect x="215" y="190" width="310" height="110" rx="10" fill="#2272B4" fill-opacity="0.08" stroke="#2272B4" stroke-width="1.5"/>
  <text x="370" y="218" text-anchor="middle" font-size="17" font-weight="700" fill="#2272B4">Custom RAG</text>
  <text x="370" y="238" text-anchor="middle" font-size="14" fill="#0b2026">AI SQL Functions + Additional Logic</text>
  <text x="370" y="258" text-anchor="middle" font-size="13" fill="#618794">ai_parse_document → ai_prep_search</text>
  <text x="370" y="274" text-anchor="middle" font-size="13" fill="#618794">→ Delta table → AI Search index</text>
  <text x="370" y="290" text-anchor="middle" font-size="13" fill="#618794">Full control over every component</text>

  <!-- Converge arrows -->
  <path d="M525 65 L600 135" fill="none" stroke="#00A972" stroke-width="1.8" marker-end="url(#rag-arrow-green)"/>
  <path d="M525 245 L600 175" fill="none" stroke="#2272B4" stroke-width="1.8" marker-end="url(#rag-arrow-blue)"/>

  <!-- Answers (shared end) -->
  <rect x="605" y="110" width="160" height="90" rx="8" fill="#ffffff" stroke="#1B3139" stroke-width="1.5"/>
  <text x="685" y="140" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">Answers</text>
  <text x="685" y="160" text-anchor="middle" font-size="13" fill="#618794">LLM + retrieved</text>
  <text x="685" y="176" text-anchor="middle" font-size="13" fill="#618794">context</text>
</svg>
</div>

**Managed RAG (Knowledge Assistants):**
- You provide documents (in a UC Volume or AI Search Index)
- The system handles parsing, chunking, embedding, indexing, and serving
- Optimized automatically via feedback loops
- Best for: standard RAG use cases where speed to production matters

**Custom RAG (Agent Framework + AI Search):**
- You build the pipeline yourself with full ownership of each step
- Full control over parsing strategy, chunking approach, embedding model, and retrieval logic
- Use AI Search indexes as retrieval tools in your custom agent
- Best for: unique requirements, complex multi-step reasoning, novel architectures

### B2. Governance with Unity Catalog

AI Search indexes are governed by Unity Catalog. Access to Unity Catalog objects in the retrieval pipeline is controlled with Unity Catalog privileges, while AI search endpoints are controlled with endpoint ACLs. Audit visibility is available through system tables, and lineage can be captured across Unity Catalog tables and paths used in the retrieval pipeline.

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 820 200" role="img" style="font-family: sans-serif;">
 <title>Governance across the retrieval pipeline with Unity Catalog and AI search endpoint ACLs</title>
 <defs>
 <marker id="uc-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
 <path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
 </marker>
 </defs>

 <!-- Governance banner -->
 <rect x="10" y="10" width="800" height="180" rx="10" fill="#F9F7F4" stroke="#1B3139" stroke-width="1" stroke-opacity="0.3"/>
 <text x="410" y="32" text-anchor="middle" font-size="14" font-weight="700" fill="#618794">Governance across the retrieval pipeline</text>

 <!-- UC Volume -->
 <rect x="30" y="58" width="120" height="56" rx="8" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.2"/>
 <text x="90" y="82" text-anchor="middle" font-size="14" font-weight="600" fill="#0b2026">UC Volume</text>
 <text x="90" y="100" text-anchor="middle" font-size="12" fill="#618794">Source PDFs</text>

 <path d="M150 86 L185 86" fill="none" stroke="#1B3139" stroke-width="1.5" marker-end="url(#uc-arrow)"/>

 <!-- Delta Table -->
 <rect x="190" y="58" width="145" height="56" rx="8" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.2"/>
 <text x="262.5" y="82" text-anchor="middle" font-size="14" font-weight="600" fill="#0b2026">Delta Table</text>
 <text x="262.5" y="100" text-anchor="middle" font-size="12" fill="#618794">Chunks + metadata</text>

 <path d="M335 86 L370 86" fill="none" stroke="#1B3139" stroke-width="1.5" marker-end="url(#uc-arrow)"/>

 <!-- VS Index -->
 <rect x="375" y="58" width="135" height="56" rx="8" fill="#00A972" fill-opacity="0.10" stroke="#00A972" stroke-width="1.2"/>
 <text x="442.5" y="82" text-anchor="middle" font-size="14" font-weight="600" fill="#0b2026">VS Index</text>
 <text x="442.5" y="100" text-anchor="middle" font-size="12" fill="#618794">Vectors + synced cols</text>

 <path d="M510 86 L555 86" fill="none" stroke="#1B3139" stroke-width="1.5" marker-end="url(#uc-arrow)"/>

 <!-- VS Endpoint -->
 <rect x="560" y="58" width="120" height="56" rx="8" fill="#2272B4" fill-opacity="0.08" stroke="#2272B4" stroke-width="1.2"/>
 <text x="620" y="82" text-anchor="middle" font-size="14" font-weight="600" fill="#0b2026">VS Endpoint</text>
 <text x="620" y="100" text-anchor="middle" font-size="12" fill="#618794">Query interface</text>

 <path d="M680 86 L715 86" fill="none" stroke="#1B3139" stroke-width="1.5" marker-end="url(#uc-arrow)"/>

 <!-- KA / Agent -->
 <rect x="720" y="58" width="70" height="56" rx="8" fill="#2272B4" fill-opacity="0.10" stroke="#2272B4" stroke-width="1.2"/>
 <text x="755" y="82" text-anchor="middle" font-size="14" font-weight="600" fill="#0b2026">Agent</text>
 <text x="755" y="100" text-anchor="middle" font-size="12" fill="#618794">RAG</text>

 <!-- Governance labels -->
 <g>
 <text x="90" y="132" text-anchor="middle" font-size="12" fill="#618794">UC privileges</text>
 <text x="262.5" y="132" text-anchor="middle" font-size="12" fill="#618794">UC privileges</text>
 <text x="442.5" y="132" text-anchor="middle" font-size="12" fill="#618794">UC privileges</text>
 <text x="620" y="132" text-anchor="middle" font-size="12" fill="#618794">Endpoint ACLs</text>
 <text x="755" y="125" text-anchor="middle" font-size="12" fill="#618794">Queries via</text>
 <text x="755" y="139" text-anchor="middle" font-size="12" fill="#618794">endpoint</text>

</svg>
</div>

<div style="width: 100%; font-family: sans-serif;">

<div style="background: #F9F7F4; border-radius: 10px; padding: 24px 28px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">

  <img src="./Includes/images/genie-code.png" style="height: 64px; margin-bottom: 10px;">

  <div style="font-size: 15pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">
    Want to learn more about AI Search and embedding strategies? Ask Genie Code. Click on the genie icon <img src="./Includes/images/genie-icon.png" style="height: 32px; vertical-align: middle;"> and begin querying. For example, click the <strong>Copy</strong> button below and paste into <strong>Genie Code</strong>.
  </div>

  <div style="display: flex; align-items: center; gap: 10px; background: #fff; border: 1px solid #ddd; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0b2026;">
    <span id="genie-query" style="flex: 1;">How does Databricks AI Search handle hybrid keyword and semantic search?</span>
    <button onclick="
      var text = document.getElementById('genie-query').innerText;
      var ta = document.createElement('textarea');
      ta.value = text;
      ta.style.position = 'fixed';
      ta.style.opacity = '0';
      document.body.appendChild(ta);
      ta.select();
      document.execCommand('copy');
      document.body.removeChild(ta);
      this.innerText = 'Copied!';
      var btn = this;
      setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
    " style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 4px 12px; font-size: 13pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>

</div>

</div>

## C. Conclusion

In this lecture, we covered:

1. **Embeddings** transform text into vectors that capture semantic meaning, which is what your index uses to match queries to chunks
2. **Cosine similarity** ranks results by how close their vectors are to your query vector
3. **Knowledge Assistants** are the managed RAG path. They run the same parse/chunk/embed/index pipeline automatically
4. **Custom RAG** gives you full control when the managed path doesn't fit your needs
5. **Unity Catalog** governs every component in the pipeline with ACLs, lineage, and audit

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
